# 1. Load table bronze.stops

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row, Column

In [0]:
df = spark.read.table("warsaw_gtfs.bronze.stops")
df = df.withColumn('row_id', F.monotonically_increasing_id())

print(df.count())
# stops_df.printSchema()
df.display()
 



In [0]:

# Data issues
# 1. Not friendly child stop_names (done)
# 2. Empty Platform codes for metro stops except swietokrzyska- M1 for metro 1 and M2 for metro 2 (done)
# 3. Null city name values, default to empty string (done)
# 4. Encoded location_type names (done)
# 5. Encoded wheelchair_boarding values (done)
# 6. 

#2. Transformations

##Fill null town names with Warszawa as null values contain only metro stations in warsaw


In [0]:
# fill null town_names with Warszawa

df_ftn= df.na.fill('Warszawa', ['town_name'])

# check if any town_name is null
# stops_df_filled_town_name.where('town_name is null').display()
df_ftn.display()

##Make numeric/child stop names more friendly

In [0]:
# identify numeric child stop_names
df_flagged = df_ftn.withColumn(
    'is_numeric_stop_name',
    F.col('stop_name').rlike('^[0-9]*$')
)
#df_flagged.limit(50).display()

# create a parent name lookup table
df_parents = df_ftn.where('parent_station is null').select(
    F.col('stop_id').alias('parent_stop_id'),
    F.col('stop_name').alias('parent_stop_name')
)
#df_parents.limit(50).display()

# join the parent table with the dataframe
df_joined = df_flagged.join(df_parents, df_flagged.parent_station == df_parents.parent_stop_id, how = 'left')

# make a final table with the name replaced with the more friendly name
df_final = df_joined.withColumn(
    'stop_name',
    F.when(
        F.col('is_numeric_stop_name'),
        F.concat(F.col('parent_stop_name'), F.lit(' '), F.col('stop_name'))
        ).otherwise(F.col('stop_name'))
)
# drop unnessecary columns
df_final = df_final.drop('parent_stop_id', 'parent_stop_name', 'is_numeric_stop_name')
df_final.limit(50).display()

##Map location_type to more friendly values

In [0]:
df_final = df_final.withColumn(
    'location_type',
    F.when((F.col('location_type') == 0) & (F.col('stop_id').rlike('^.+P.*')), 'metro platform')
    .when(F.col('location_type') == 0, 'distinct stop')
    .when(F.col('location_type') == 1, 'parent stop')
    .otherwise('child stop')
)

df_final.filter(F.col('location_type') == 'metro platform').display()

##Map wheelchair to more friendly values

In [0]:
df_final2 = df_final.withColumn(
    'wheelchair_boarding',
    F.when(F.col('wheelchair_boarding') == 0, 'no info')
     .when(F.col("wheelchair_boarding") == 1, 'yes')
     .otherwise('no')
)
df_final2.filter(F.col('wheelchair_boarding') == 'no').display()

##Mark null platform codes for metro stations


In [0]:

df_final2 = df_final2.withColumn(
    'platform_code',
    F.when(F.col('stop_id').rlike('^.+P1*$'), 'M1')
    .when(F.col('stop_id').rlike('^.+P2*$'), 'M2')
    .otherwise('n/a')
)


In [0]:
df_final2.display()

##Fill nulls with n/a

In [0]:
df_final2 = df_final2.na.fill('n/a')

#3. Write processed table to silver layer

In [0]:
df_final2.write.format('delta').mode('overwrite').saveAsTable('warsaw_gtfs.silver.stops')